In [ ]:
!apt-get update -y
!apt-get install -y proj-bin proj-data
!pip -q install geopandas shapely openpyxl
!pip install osmnx geopandas folium



In [ ]:
!pip install geopandas
!pip install folium
!pip install shapely
!pip install geovoronoi

In [ ]:
!apt-get update -y
!apt-get install -y proj-bin proj-data

In [ ]:
import pandas as pd
import geopandas as gpd
import folium
from shapely.geometry import Polygon
from scipy.spatial import Voronoi
import numpy as np
import osmnx as ox
from geovoronoi import voronoi_regions_from_coords



In [ ]:
#load school data file - list of WL primary schools
school_locations_path = '/content/west_lindsey_schools_list.csv'
df = pd.read_csv(school_locations_path)



In [ ]:
#create dataframe with Point geometries - longitude and latitude
#use GeoDataFrame : has a geometry column for locaton data. Stores type of geometrc object i.e. pont and coordinates of object
##crs - 'coordinate refernce system'. EPSG-'European Petroleum Survey Group'. '4326' - code for GPS Global Positioning System

##GPS projects the latitude and longitude coordinates from 3D planet onto 2D flat surface of map uing UK British National Grid CRS

gdf = gpd.GeoDataFrame(df, geometry = gpd.points_from_xy(df['Longitude'], df['Latitude']), crs = 'EPSG:4326')



In [ ]:
#project to meters for flat Voronoi grid
gdf_m = gdf.to_crs("EPSG:27700")

#new projected coordinates
coords = np.column_stack((gdf_m.geometry.x, gdf_m.geometry.y))
print (min(gdf_m.geometry.x), max(gdf_m.geometry.x))
print (min(gdf_m.geometry.y), max(gdf_m.geometry.y))
#print (coords)

In [ ]:
#create Voronoi grid by passng 'longitude' and 'latitude' from gdf to Voronoi class

vor = Voronoi(coords) #using coorindates of a flat map instead


In [ ]:
#boundary created around WL mapping (finds the boundary box coords determined by the mapping of WL (instead of an overall estimated boundry box))
#use ox to model and visualise the geospatial features of Calderdale from OpenStreetMap
WL_geometry = ox.geocode_to_gdf("West Lindsey, Lincolnshire, England, UK")

print(list(WL_geometry.columns))

WL_boundary_pre = WL_geometry.geometry.iloc[0] #access only the geometry column in the calderdale boundary information list
WL_boundary = gpd.GeoSeries([WL_boundary_pre], crs='EPSG:4326').to_crs('EPSG:27700')[0]



In [ ]:
#for voronoi:
cell_polygons, cell_coords = voronoi_regions_from_coords(coords, WL_boundary)
voronoi_polygons = list(cell_polygons.values())

gdf_m_voronoi = gpd.GeoDataFrame(geometry=voronoi_polygons, crs='EPSG:27700')

print (gdf_m_voronoi)


In [ ]:
#make list of Voronoi polygons and remove invalid regions (empty or extend to infinity)
##if 'region' attribute of vor (from Voronoi() class) starts wth '-1' the regon never closes
##pass vertices attribute to shapely's Polygon() class. Generates plottable polygons

#PREV CODE: voronoi_polygons = [Polygon(vor.vertices[region]) for region in vor.regions if region and -1 not in region]

#create GeoDataFrame with Voronoi polygons
#gdf_voronoi = gpd.GeoDataFrame(geometry = voronoi_polygons, crs='EPSG:4326')


#Apply truncation to limit area to within Calderdale Area

# Define the bounding box lat-lon limits: - ALTER MAX AND MIN LAT AND LONG VALS AS REQUIRED
#max_lat, min_lat, max_lon, min_lon = (53.9, 53.5, -1.8, -2.25)

##min_x, min_y, max_x, max_y = gdf_m.total_bounds #create bounds from data vals instead of lat and long vals
##pad = 5000 #5k distance as padding for bounds

# Create the bounding box as a Shapely Polygon
#####bounding_box = Polygon.from_bounds(min_lon, min_lat, max_lon, max_lat)
##bounding_box_m = Polygon.from_bounds(min_x - pad, min_y - pad, max_x + pad, max_y + pad)

# Truncate each Voronoi polygon with the bounding box:
#####truncated_polygons = [polygon.intersection(bounding_box) for polygon in gdf_voronoi.geometry]
##truncated_polygons_m = [polygon.intersection(bounding_box_m) for polygon in gdf_m_voronoi.geometry]


# Create a GeoDataFrame with the (truncated) voronoi polygons:
#####gdf_truncated = gpd.GeoDataFrame(geometry=truncated_polygons, crs='EPSG:4326')
##gdf_m_truncated_pre = gpd.GeoDataFrame(geometry=truncated_polygons_m,crs='EPSG:27700') #coordinates are in metres for planar projection, better accuracy to calculae voronoi


#convert polygons back to longitude and latitude
##gdf_m_truncated = gdf_m_truncated_pre.to_crs('EPSG:4326') #better to transform back to 4326 for visualisation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Plot map - using Folium to draw it over a street map of the city
## -> choose openstreetmap as the tiles argument
## -> control icon inside marker by setting the icon symbol

#convert polygons back to longitude and latitude
gdf_m_voronoi_plot = gdf_m_voronoi.to_crs('EPSG:4326')

#folium map centred on average coordinates of the schools:
map_centre = [gdf['Latitude'].mean(), gdf['Longitude'].mean()]
school_map = folium.Map(location = map_centre, zoom_start =12, tiles = 'OpenStreetmap')

#plot Voronoi polygons on the map
folium.GeoJson(gdf_m_voronoi_plot).add_to(school_map)

#add markers for each school
#for index, school in gdf.iterrows():
#  folium.Marker(location = [school['Latitude'], school['Longitude']], popup = f"{school['Establishment']}\n{school['Address 1']}\n{school['Address 2']}\n{school['Address 4']}{school['Postcode']}", icon=folium.Icon(color='blue', icon='home')).add_to(school_map)

#display map
school_map #displays primary and secondary map

In [ ]:
##To further develop: can blend other limitations/ incporporate other boundaries i.e. closest transport-wise instead of purely nearest neighbour into the determining of the Voronoi

#Extract the boundaries of the voronoi cells as seperate coordinates.

gdf_m_voronoi["areas_m2"] = gdf_m_voronoi.geometry.area
gdf_m_voronoi["areas_km2"] = gdf_m_voronoi["areas_m2"]/ 1e6

#find the polygon ccoordinates for each school 'cell' representing the schools catchment area. cell_iD is in order of the primary schools as they were entered.
#pairing these polygon coordinates to their respective schools that they represent the boundary for
cell_rows = []
for cell_id, geom in enumerate(gdf_m_voronoi.geometry):
  if geom is None or geom.is_empty:
    continue
  if geom.geom_type == "Polygon":
    polygons = [geom]
  else:
    continue
    #compiing all geometrical polygon data for each school catchment area - longitude, latitude, area (km). Can be accesses through the schools corresponding cell_id
  cell_area_km2 = gdf_m_voronoi.iloc[cell_id]["areas_km2"]

  for i, polygon in enumerate(polygons):
    xys = list(polygon.exterior.coords)

    for j, (x,y) in enumerate(xys):
      cell_rows.append({"cell_id": cell_id, "polygon_index": i, "vertex_index": j, "x": x, "y": y, 'area_km2': cell_area_km2})

df_vertices = pd.DataFrame(cell_rows)

print(df_vertices)


In [ ]:
#isolate the areas of the schools. identify schools through their cell_id
cell_id_area = df_vertices[["cell_id", "area_km2"]].drop_duplicates()

In [ ]:
#Save to output excel file
out_path = "/content/WL_voronoi_cells_areas.xlsx"
cell_id_area.to_excel(out_path, index=False)
out_path

In [ ]:
#add West Lindsey flood zones layer

import os

#style_function and tooltip corresponds with that used in seperate flood risk zones map for continuity and easier readability
def style_function(feature):
    zone = feature["properties"].get("flood_zone", "")
    if zone == "FZ2":
        color = "#1f78b4"  # blue
    elif zone == "FZ3":
        color = "#bd0026"  # red
    else:
        color = "#8c8c8c"  # grey fallback
    return {
        "fillColor": color,
        "color": color,
        "weight": 0.5,
        "fillOpacity": 0.4,
        "opacity": 0.6,
    }

tooltip = folium.GeoJsonTooltip(
    fields=["flood_zone"],
    aliases=["Flood Zone"],
    sticky=False,
)


folium.GeoJson("westlindsey_floodzones_2_3.geojson", name='Flood Zones', style_function=style_function, tooltip=tooltip).add_to(school_map)

school_map




In [ ]:
#save file as html

school_map.save("layered_WL_skl_flood_risk_map.html")

In [ ]:
#double-check dataset
print (gdf_m_voronoi)

In [ ]:
#percentage of each cell area under high flood risk

#find overlay of flood areas in each voronoi cell
##(Road networks aspect covered seperately - this is purely flood zone areas)

#seperate high risk areas:
flood_risk_areas = gpd.read_file("westlindsey_floodzones_2_3.geojson").query("flood_zone == 'FZ3' ")

#match crs to catchment area crs
flood_risk_areas = flood_risk_areas.to_crs(gdf_m_voronoi.crs)

print (flood_risk_areas)


In [ ]:
# find intersections between catchment areas and high flood risks

#assign cell_id to each catchment area
gdf_m_voronoi["cell_id"] = gdf_m_voronoi.index

#removes any buffer zones around the catchment area cells and the flood risk area polygons -
gdf_m_voronoi["geometry"] = gdf_m_voronoi.geometry.buffer(0)
flood_risk_areas["geometry"] = flood_risk_areas.geometry.buffer(0)

#finds the voronoi cell which overlaps each flood risk area polygon
intersections = gpd.overlay(gdf_m_voronoi[["cell_id", "geometry" ]], flood_risk_areas[["geometry"]], how="intersection")

# Areas
intersections["overlap_area"] = intersections.area

# Sum overlap per cell - sums the different flood risk area polygons within each catchment area - total flood risk within a catchment area
overlap_per_cell = (intersections.groupby("cell_id")["overlap_area"].sum())

#fill NaN vals with 0:
gdf_m_voronoi["flood_risk_areas"] = (gdf_m_voronoi["cell_id"].map(overlap_per_cell).fillna(0))

print (gdf_m_voronoi)

# Percentage
gdf_m_voronoi["pct_high_flood"] = (gdf_m_voronoi["flood_risk_areas"] / gdf_m_voronoi["areas_m2"]) * 100




In [ ]:
#Save % flood risk areas to output excel file
WL_flood_risk_out_path = "/content/WL_PCT_FLOOD_RISK.xlsx"
cell_id_area.to_excel(WL_flood_risk_out_path, index=False)
WL_flood_risk_out_path

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10,8))

gdf_m_voronoi.plot(
    column="pct_high_flood",
    ax=ax,
    legend=True,
    edgecolor='black',
    linewidth=0.5,
    legend_kwds={
        "shrink": 0.6,
        "label": "% High Flood Risk Area"
    }
)

# colour bar
cbar = ax.get_figure().axes[-1]
cbar.set_ylabel("% High Flood Risk Area", fontsize=24) # Colour bar label
cbar.tick_params(labelsize=21) # Colour bar tick size

# Title size
ax.set_title("% of high flood risk per school catchment area",fontsize=25)
ax.axis("off")

#Save as PDF
plt.savefig("WL_voronoi_flood_percent_map.pdf",bbox_inches="tight")

plt.show()
plt.close()